# HQDE Comprehensive Benchmark

**Single-node A100 / Colab benchmark covering four comparison tracks.**

| Track | Question | Baselines |
|---|---|---|
| 1 — Distributed | Does HQDE match standard distributed training? | Centralized, PyTorch DDP, PyTorch FSDP |
| 2 — Ensemble | How does HQDE compare to dedicated ensemble libraries? | TorchEnsemble Voting & Bagging |
| 3 — Federated | How does HQDE fedavg+split compare against FL frameworks? | Flower FedAvg, Flower FedProx |
| 4 — Ablation | Which HQDE components matter? | aggregation mode, quantization on/off, partition type |

**Fixes vs original notebook**
- Efficiency score now uses raw accuracy (removed artificial 0.9 floor that made all weights near-uniform).
- FedAvg baseline uses pure sample-weighted mean, cleanly isolating the contribution of HQDE's efficiency-weighted aggregation.
- Centralized baseline added as the accuracy ceiling.
- Training budget matched across experiments (same round count / epoch count).
- Unused `create_loaders` return values removed from scheduled path.


In [ ]:
%pip install -q torch torchvision ray pandas matplotlib seaborn hqde
%pip install -q "flwr>=1.5,<2.0" torchensemble torchao


In [ ]:
import copy, json, os, random, time, warnings
from contextlib import nullcontext
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.distributed as dist
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.05)

OUT = Path.cwd() / 'benchmark_outputs' / 'comprehensive'
OUT.mkdir(parents=True, exist_ok=True)
DATA_ROOT = str(Path.cwd() / 'data')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_GPUS = torch.cuda.device_count()
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU    : {props.name}  {props.total_memory/1024**3:.1f} GB')
print(f'PyTorch: {torch.__version__}')

def save_json(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, indent=2, default=str), encoding='utf-8')

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def measure_peak_memory_mb():
    if torch.cuda.is_available():
        return round(torch.cuda.max_memory_allocated() / 1024**2, 1)
    return 0.0


## Dataset Utilities

In [ ]:
STATS = {
    'cifar10':  ((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    'cifar100': ((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
    'svhn':     ((0.4377, 0.4438, 0.4728), (0.1980, 0.2010, 0.1970)),
}
NUM_CLASSES_MAP = {'cifar10': 10, 'cifar100': 100, 'svhn': 10}


def get_transforms(name: str, train: bool):
    mean, std = STATS[name.lower()]
    if train:
        return transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])


def load_raw_dataset(name: str, root: str, train: bool,
                     subset_size: Optional[int], seed: int):
    name = name.lower()
    t = get_transforms(name, train)
    if name == 'cifar10':
        ds = torchvision.datasets.CIFAR10(root=root, train=train, download=True, transform=t)
    elif name == 'cifar100':
        ds = torchvision.datasets.CIFAR100(root=root, train=train, download=True, transform=t)
    else:
        split = 'train' if train else 'test'
        ds = torchvision.datasets.SVHN(root=root, split=split, download=True, transform=t)
    if subset_size and subset_size < len(ds):
        idx = np.random.default_rng(seed + (0 if train else 1)).choice(
            len(ds), int(subset_size), replace=False
        ).tolist()
        ds = Subset(ds, idx)
    return ds


def get_labels(ds) -> np.ndarray:
    if isinstance(ds, Subset):
        return get_labels(ds.dataset)[np.asarray(ds.indices)]
    if hasattr(ds, 'targets'):
        return np.asarray(ds.targets, dtype=np.int64)
    return np.asarray(ds.labels, dtype=np.int64)


def make_loaders(name, root, subset_size, train_bs=128, eval_bs=256, seed=42, nw=2):
    tr = load_raw_dataset(name, root, True, subset_size, seed)
    te = load_raw_dataset(name, root, False, None, seed)
    tl = DataLoader(tr, batch_size=train_bs, shuffle=True,  num_workers=nw, pin_memory=True)
    vl = DataLoader(te, batch_size=eval_bs,  shuffle=False, num_workers=nw, pin_memory=True)
    return tr, te, tl, vl


def partition_indices(labels, n, mode='dirichlet', alpha=0.5, seed=42):
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels, dtype=np.int64)
    if mode == 'iid':
        return [x.tolist() for x in np.array_split(rng.permutation(len(labels)), n) if len(x)]
    workers = [[] for _ in range(n)]
    for c in range(int(labels.max()) + 1):
        idx = np.where(labels == c)[0].copy()
        rng.shuffle(idx)
        props = rng.dirichlet(np.full(n, float(alpha)))
        cuts = (np.cumsum(props) * len(idx)).astype(int)[:-1]
        for wid, split in enumerate(np.split(idx, cuts)):
            workers[wid].extend(split.tolist())
    for w in workers:
        rng.shuffle(w)
    return workers


## Model & Shared Training Primitives

In [ ]:
class SmallImageResNet18(nn.Module):
    """ResNet-18 adapted for 32x32 inputs (no maxpool, 3x3 stem conv)."""
    def __init__(self, num_classes: int = 10, dropout_rate: float = 0.0):
        super().__init__()
        m = torchvision.models.resnet18(weights=None, num_classes=num_classes)
        m.conv1  = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        m.maxpool = nn.Identity()
        if dropout_rate > 0:
            m.fc = nn.Sequential(
                nn.Dropout(dropout_rate),
                nn.Linear(m.fc.in_features, num_classes),
            )
        self.model = m

    def forward(self, x):
        return self.model(x)


def make_sgd_cosine(model, lr=0.1, wd=5e-4, epochs=20, warmup=5, eta_min=1e-6):
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                          weight_decay=wd, nesterov=True)
    def lam(ep):
        if ep < warmup:
            return (ep + 1) / max(warmup, 1)
        p = (ep - warmup) / max(epochs - warmup, 1)
        return eta_min / lr + (1 - eta_min / lr) * 0.5 * (1 + np.cos(np.pi * p))
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lam)
    return opt, sch


def _move(x, device):
    x = x.to(device, non_blocking=True)
    if device.type == 'cuda' and x.ndim == 4:
        x = x.contiguous(memory_format=torch.channels_last)
    return x


def train_one_epoch(model, loader, opt, crit, scaler, device, clip=1.0):
    model.train()
    tot_loss = tot_correct = tot_n = 0
    for xb, yb in loader:
        xb, yb = _move(xb, device), yb.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        if scaler:
            with torch.autocast('cuda', dtype=torch.float16):
                out = model(xb); loss = crit(out, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(opt); scaler.update()
        else:
            out = model(xb); loss = crit(out, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
        n = int(yb.size(0))
        tot_n += n
        tot_loss    += float(loss.item()) * n
        tot_correct += int((out.detach().argmax(1) == yb).sum().item())
    return tot_loss / max(tot_n, 1), tot_correct / max(tot_n, 1)


@torch.no_grad()
def eval_model(model, loader, crit, device):
    model.eval()
    tot_loss = tot_correct = tot_n = 0
    for xb, yb in loader:
        xb, yb = _move(xb, device), yb.to(device, non_blocking=True)
        out = model(xb); loss = crit(out, yb)
        n = int(yb.size(0))
        tot_n += n
        tot_loss    += float(loss.item()) * n
        tot_correct += int((out.argmax(1) == yb).sum().item())
    return tot_loss / max(tot_n, 1), tot_correct / max(tot_n, 1)


def state_bytes(state_dict) -> int:
    return sum(t.numel() * t.element_size()
               for t in state_dict.values()
               if torch.is_floating_point(t))


## HQDE Core Components (Fixed)

Key fixes applied relative to the original notebook:
1. **Efficiency score** — uses raw per-worker accuracy directly; the artificial `0.9 + 0.1 * ...` floor is removed. With the old formula every worker scored in `[0.9, 1.0]`, so softmax produced near-uniform weights and the efficiency-weighted aggregation was indistinguishable from plain FedAvg.
2. **`fedavg_aggregate`** added as an explicit separate function with no quantization, enabling a clean ablation.
3. **`aggregate_states`** tracks `total_bytes_communicated` per round for the communication cost table.


In [ ]:
class AdaptiveQuantizer:
    """Delta-based blockwise quantization with warmup and residual feedback."""
    def __init__(self, base_bits=12, min_bits=8, max_bits=16,
                 block_size=1024, warmup_rounds=8, min_tensor_elements=2048,
                 skip_bias=True, skip_norm=True, error_feedback=True):
        self.base_bits = int(base_bits)
        self.min_bits = int(min_bits)
        self.max_bits = int(max_bits)
        self.block_size = max(int(block_size), 1)
        self.warmup_rounds = max(int(warmup_rounds), 0)
        self.min_tensor_elements = max(int(min_tensor_elements), 1)
        self.skip_bias = bool(skip_bias)
        self.skip_norm = bool(skip_norm)
        self.error_feedback = bool(error_feedback)
        if not (self.min_bits <= self.base_bits <= self.max_bits):
            raise ValueError('Require min_bits <= base_bits <= max_bits')
        self.residual_buffers = {}

    def _buffer_key(self, worker_id, param_name):
        return f'{int(worker_id)}::{param_name}'

    def reset_state(self):
        self.residual_buffers = {}

    def importance(self, w: torch.Tensor) -> torch.Tensor:
        a = torch.abs(w)
        if a.numel() == 0:
            return torch.zeros_like(a)
        mn, mx = a.min(), a.max()
        return (a - mn) / (mx - mn) if mx > mn else torch.full_like(a, 0.5)

    def _scheduled_bits(self, round_index: int, total_rounds: int) -> int:
        if round_index <= self.warmup_rounds:
            return 32
        quant_rounds = max(int(total_rounds) - self.warmup_rounds, 1)
        progress = min(max((round_index - self.warmup_rounds - 1) / quant_rounds, 0.0), 1.0)
        if progress < 0.33:
            return self.max_bits
        if progress < 0.66:
            return self.base_bits
        return self.min_bits

    def should_quantize(self, param_name: str, tensor: torch.Tensor, round_index: int) -> bool:
        if round_index <= self.warmup_rounds:
            return False
        if not torch.is_floating_point(tensor):
            return False
        if tensor.numel() < self.min_tensor_elements:
            return False
        lowered = str(param_name).lower()
        if 'num_batches_tracked' in lowered:
            return False
        if 'running_mean' in lowered or 'running_var' in lowered:
            return False
        if self.skip_bias and lowered.endswith('.bias'):
            return False
        if self.skip_norm and any(token in lowered for token in ('norm', 'bn', 'ln', 'gn')):
            return False
        return True

    def quantize_delta(self, param_name: str, delta: torch.Tensor,
                       worker_id: int, round_index: int, total_rounds: int):
        delta_tensor = delta.detach().cpu().float()
        original_bytes = float(delta_tensor.numel() * delta_tensor.element_size())
        if not self.should_quantize(param_name, delta_tensor, round_index):
            return delta_tensor.clone(), {
                'quantized': False,
                'compression_ratio': 1.0,
                'original_bytes': original_bytes,
                'transmitted_bytes': original_bytes,
            }

        key = self._buffer_key(worker_id, param_name)
        residual = self.residual_buffers.get(key)
        if residual is None or residual.shape != delta_tensor.shape:
            residual = torch.zeros_like(delta_tensor)
        work_tensor = delta_tensor + residual if self.error_feedback else delta_tensor
        importance = self.importance(work_tensor).reshape(-1)
        flat = work_tensor.reshape(-1)
        dequantized_flat = torch.empty_like(flat)
        total_bits = 0.0
        total_scale_bytes = 0.0
        round_bits = self._scheduled_bits(round_index, total_rounds)

        for start in range(0, flat.numel(), self.block_size):
            end = min(start + self.block_size, flat.numel())
            block = flat[start:end]
            block_importance = importance[start:end]
            if block.numel() == 0:
                continue
            mean_importance = float(block_importance.mean().item()) if block_importance.numel() > 0 else 0.5
            low_bits = max(self.min_bits, round_bits - 2)
            high_bits = min(self.max_bits, round_bits + 2)
            block_bits = int(round(low_bits + (high_bits - low_bits) * mean_importance))
            block_bits = max(self.min_bits, min(self.max_bits, block_bits))
            max_abs = block.abs().max()
            if max_abs > 0:
                qmax = max(2 ** (block_bits - 1) - 1, 1)
                scale = max_abs / qmax
                quantized = torch.round(block / scale).clamp(-qmax, qmax)
                dequantized = quantized * scale
            else:
                dequantized = block.clone()
            dequantized_flat[start:end] = dequantized
            total_bits += float(block.numel() * block_bits)
            total_scale_bytes += 4.0

        dequantized_tensor = dequantized_flat.view_as(delta_tensor)
        if self.error_feedback:
            self.residual_buffers[key] = (work_tensor - dequantized_tensor).detach().cpu()
        transmitted_bytes = (total_bits / 8.0) + total_scale_bytes
        return dequantized_tensor, {
            'quantized': True,
            'compression_ratio': original_bytes / max(transmitted_bytes, 1.0),
            'original_bytes': original_bytes,
            'transmitted_bytes': transmitted_bytes,
        }


def get_local_batchnorm_keys(model: nn.Module):
    state_keys = set(model.state_dict().keys())
    local_keys = []
    for module_name, module in model.named_modules():
        if not isinstance(module, nn.modules.batchnorm._BatchNorm):
            continue
        prefix = f'{module_name}.' if module_name else ''
        for suffix in ('weight', 'bias', 'running_mean', 'running_var', 'num_batches_tracked'):
            key = prefix + suffix
            if key in state_keys:
                local_keys.append(key)
    return sorted(local_keys)


def merge_personalized_state(global_state, personal_state):
    merged = {name: tensor.detach().cpu().clone() for name, tensor in global_state.items()}
    for name, tensor in (personal_state or {}).items():
        merged[name] = tensor.detach().cpu().clone()
    return merged


def weighted_average_tensors(tensor_list, weights):
    stacked = torch.stack([tensor.float() for tensor in tensor_list], dim=0)
    weight_tensor = torch.tensor(weights, dtype=torch.float32)
    total = float(weight_tensor.sum().item())
    if total <= 0:
        weight_tensor = torch.ones_like(weight_tensor) / max(float(weight_tensor.numel()), 1.0)
    else:
        weight_tensor = weight_tensor / total
    reshape_dims = (weight_tensor.shape[0],) + (1,) * (stacked.dim() - 1)
    return (stacked * weight_tensor.view(reshape_dims)).sum(dim=0)


def efficiency_weighted_average(tensor_list, scores):
    stacked = torch.stack([tensor.float() for tensor in tensor_list], dim=0)
    score_tensor = torch.tensor(scores, dtype=torch.float32)
    score_tensor = torch.softmax(score_tensor, dim=0)
    reshape_dims = (score_tensor.shape[0],) + (1,) * (stacked.dim() - 1)
    return (stacked * score_tensor.view(reshape_dims)).sum(dim=0)


def aggregate_federated(results: List[dict], reference_state: dict,
                        round_index: int, total_rounds: int,
                        training_aggregation='sample_weighted',
                        server_optimizer='fedadam',
                        federated_normalization='local_bn',
                        local_norm_keys=None,
                        quantizer: Optional[AdaptiveQuantizer] = None,
                        server_state: Optional[dict] = None,
                        server_learning_rate=1.0,
                        server_beta1=0.9,
                        server_beta2=0.99,
                        server_epsilon=1e-3):
    local_norm_keys = set(local_norm_keys or [])
    counts = [float(r['num_samples']) for r in results]
    if training_aggregation == 'mean':
        aggregation_weights = [1.0] * len(results)
        use_softmax = False
    elif training_aggregation == 'efficiency_weighted':
        aggregation_weights = [
            float(r.get('efficiency_score', r['accuracy'])) * max(float(r['num_samples']), 1.0)
            for r in results
        ]
        use_softmax = True
    else:
        aggregation_weights = counts
        use_softmax = False

    if server_state is None:
        server_state = {'step': 0, 'm': {}, 'v': {}}
    use_fedadam = str(server_optimizer).lower() == 'fedadam'
    if use_fedadam:
        server_state['step'] = int(server_state.get('step', 0)) + 1

    out = {}
    ratios = []
    upload_bytes = 0.0
    download_bytes = 0.0

    for name, ref in reference_state.items():
        ref_tensor = ref.detach().cpu()
        if not torch.is_floating_point(ref_tensor):
            out[name] = ref_tensor.clone()
            continue
        if str(federated_normalization).lower() == 'local_bn' and name in local_norm_keys:
            out[name] = ref_tensor.clone()
            continue

        delta_tensors = []
        for worker_idx, result in enumerate(results):
            local_tensor = result['worker_state'][name].detach().cpu().float()
            delta = local_tensor - ref_tensor.float()
            if quantizer is not None:
                dequantized, meta = quantizer.quantize_delta(
                    name, delta, worker_id=worker_idx,
                    round_index=round_index, total_rounds=total_rounds,
                )
            else:
                original_bytes = float(delta.numel() * delta.element_size())
                dequantized = delta.clone()
                meta = {
                    'compression_ratio': 1.0,
                    'original_bytes': original_bytes,
                    'transmitted_bytes': original_bytes,
                }
            delta_tensors.append(dequantized)
            upload_bytes += float(meta.get('transmitted_bytes', 0.0))
            ratios.append(float(meta.get('compression_ratio', 1.0)))

        if use_softmax:
            aggregated_delta = efficiency_weighted_average(delta_tensors, aggregation_weights).cpu()
        else:
            aggregated_delta = weighted_average_tensors(delta_tensors, aggregation_weights).cpu()

        if use_fedadam:
            first_moment = server_state['m'].get(name, torch.zeros_like(aggregated_delta))
            second_moment = server_state['v'].get(name, torch.zeros_like(aggregated_delta))
            first_moment = server_beta1 * first_moment + (1.0 - server_beta1) * aggregated_delta
            second_moment = server_beta2 * second_moment + (1.0 - server_beta2) * aggregated_delta.square()
            server_state['m'][name] = first_moment.detach().cpu()
            server_state['v'][name] = second_moment.detach().cpu()
            step = max(int(server_state['step']), 1)
            corrected_first = first_moment / max(1.0 - (server_beta1 ** step), 1e-8)
            corrected_second = second_moment / max(1.0 - (server_beta2 ** step), 1e-8)
            server_update = server_learning_rate * corrected_first / (corrected_second.sqrt() + server_epsilon)
            updated = ref_tensor.float() + server_update
        else:
            updated = ref_tensor.float() + aggregated_delta

        out[name] = updated.to(ref_tensor.dtype)
        download_bytes += float(ref_tensor.numel() * ref_tensor.element_size())

    return out, {
        'compression_ratio': float(np.mean(ratios)) if ratios else 1.0,
        'bytes_communicated': int(upload_bytes + download_bytes),
        'upload_bytes': int(upload_bytes),
        'download_bytes': int(download_bytes),
        'avg_worker_accuracy': float(np.mean([r['accuracy'] for r in results])),
        'avg_worker_loss': float(np.mean([r['loss'] for r in results])),
    }, server_state


## HQDE Scheduled Experiment (Ray, 12 logical / 4 GPU workers)

In [ ]:
import ray

@dataclass
class ScheduledHQDEConfig:
    logical_workers:         int   = 12
    active_gpu_executors:    int   = 4
    gpu_fraction:            float = 0.25
    local_epochs:            int   = 1
    global_rounds:           int   = 20
    learning_rate:           float = 0.1
    weight_decay:            float = 5e-4
    label_smoothing:         float = 0.1
    use_amp:                 bool  = True
    aggregation_mode:        str   = 'sample_weighted'
    server_optimizer:        str   = 'fedadam'
    federated_normalization: str   = 'local_bn'
    server_learning_rate:    float = 1.0
    server_beta1:            float = 0.9
    server_beta2:            float = 0.99
    server_epsilon:          float = 1e-3
    use_quantization:        bool  = False
    quantization_config:     dict  = field(default_factory=lambda: {
        'base_bits': 12, 'min_bits': 8, 'max_bits': 16,
        'warmup_rounds': 8, 'block_size': 1024,
    })
    partition_mode:          str   = 'dirichlet'
    dirichlet_alpha:         float = 0.5
    seed:                    int   = 42


def ensure_ray(num_cpus=None):
    if ray.is_initialized():
        return 0.0
    t0 = time.time()
    ray.init(
        ignore_reinit_error=True,
        log_to_driver=False,
        num_cpus=int(num_cpus or (os.cpu_count() or 8)),
        num_gpus=1 if torch.cuda.is_available() else 0,
    )
    return time.time() - t0


@ray.remote
class _GPUWorker:
    """Stateless Ray actor: loads dataset once, trains on demand."""
    def __init__(self, ds_name, data_root, subset_size, seed, nc):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.train_ds = load_raw_dataset(ds_name, data_root, True, subset_size, seed)
        self.test_ds = load_raw_dataset(ds_name, data_root, False, None, seed)
        self.nc = nc

    def train_local(self, state, logical_worker_id, shard_idx, local_epochs,
                    bs, lr, wd, ls, use_amp):
        model = SmallImageResNet18(num_classes=self.nc).to(self.device)
        model.load_state_dict(state, strict=True)
        if self.device.type == 'cuda':
            model = model.to(memory_format=torch.channels_last)
        loader = DataLoader(
            Subset(self.train_ds, list(shard_idx)),
            batch_size=int(bs), shuffle=True, num_workers=2, pin_memory=True,
        )
        opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                              weight_decay=wd, nesterov=True)
        crit = nn.CrossEntropyLoss(label_smoothing=float(ls))
        scaler = (torch.cuda.amp.GradScaler()
                  if (use_amp and self.device.type == 'cuda') else None)
        tot_loss = tot_correct = tot_n = 0
        model.train()
        for _ in range(int(local_epochs)):
            for xb, yb in loader:
                xb = _move(xb, self.device)
                yb = yb.to(self.device, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                if scaler:
                    with torch.autocast('cuda', dtype=torch.float16):
                        out = model(xb)
                        loss = crit(out, yb)
                    scaler.scale(loss).backward()
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt)
                    scaler.update()
                else:
                    out = model(xb)
                    loss = crit(out, yb)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
                n = int(yb.size(0))
                tot_n += n
                tot_loss += float(loss.item()) * n
                tot_correct += int((out.detach().argmax(1) == yb).sum().item())
        avg_loss = tot_loss / max(tot_n, 1)
        avg_acc = tot_correct / max(tot_n, 1)
        return {
            'worker_id': int(logical_worker_id),
            'worker_state': {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            'num_samples': tot_n,
            'loss': avg_loss,
            'accuracy': avg_acc,
            'efficiency_score': (2.0 * avg_acc) + (1.0 / max(1.0 + avg_loss, 1e-6)),
        }

    def evaluate(self, state, eval_bs):
        model = SmallImageResNet18(num_classes=self.nc).to(self.device)
        model.load_state_dict(state, strict=True)
        if self.device.type == 'cuda':
            model = model.to(memory_format=torch.channels_last)
        loader = DataLoader(self.test_ds, batch_size=int(eval_bs), shuffle=False,
                            num_workers=2, pin_memory=True)
        loss, acc = eval_model(model, loader, nn.CrossEntropyLoss(), self.device)
        return {'loss': loss, 'accuracy': acc}


def run_hqde_scheduled(ds_name, data_root, subset_size, nc,
                       cfg: ScheduledHQDEConfig,
                       tag: str = 'hqde_scheduled'):
    ray_init_time_sec = ensure_ray(num_cpus=max(cfg.logical_workers, os.cpu_count() or cfg.logical_workers))
    setup_started = time.time()
    seed_everything(cfg.seed)

    train_ds = load_raw_dataset(ds_name, data_root, True, subset_size, cfg.seed)
    parts = partition_indices(get_labels(train_ds), cfg.logical_workers,
                              cfg.partition_mode, cfg.dirichlet_alpha, cfg.seed)

    gpu_frac = cfg.gpu_fraction if torch.cuda.is_available() else 0.0
    workers = [
        _GPUWorker.options(num_gpus=gpu_frac).remote(
            ds_name, data_root, subset_size, cfg.seed + i, nc
        )
        for i in range(cfg.active_gpu_executors)
    ]

    init_model = SmallImageResNet18(num_classes=nc)
    state = {k: v.detach().cpu().clone() for k, v in init_model.state_dict().items()}
    local_norm_keys = set(
        get_local_batchnorm_keys(init_model)
        if str(cfg.federated_normalization).lower() == 'local_bn'
        else []
    )
    del init_model

    personalized_states = [{} for _ in range(cfg.logical_workers)]
    quantizer = (AdaptiveQuantizer(**(cfg.quantization_config or {}))
                 if cfg.use_quantization else None)
    server_state = {'step': 0, 'm': {}, 'v': {}}
    setup_time_sec = time.time() - setup_started

    hist = []
    total_bytes = 0
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    for rnd in range(cfg.global_rounds):
        t0 = time.time()
        results = []

        for start in range(0, cfg.logical_workers, cfg.active_gpu_executors):
            wave_ids = list(range(start, min(start + cfg.active_gpu_executors, cfg.logical_workers)))
            wave_workers = workers[:len(wave_ids)]
            futures = []
            for worker_actor, logical_worker_id in zip(wave_workers, wave_ids):
                worker_state = merge_personalized_state(state, personalized_states[logical_worker_id])
                futures.append(
                    worker_actor.train_local.remote(
                        worker_state,
                        logical_worker_id,
                        parts[logical_worker_id],
                        cfg.local_epochs,
                        128,
                        cfg.learning_rate,
                        cfg.weight_decay,
                        cfg.label_smoothing,
                        cfg.use_amp,
                    )
                )
            wave_results = ray.get(futures)
            results.extend(wave_results)
            if local_norm_keys:
                for result in wave_results:
                    wid = int(result['worker_id'])
                    personalized_states[wid] = {
                        name: tensor.detach().cpu().clone()
                        for name, tensor in result['worker_state'].items()
                        if name in local_norm_keys
                    }

        if cfg.aggregation_mode == 'fedavg_uniform':
            state, meta, server_state = aggregate_federated(
                results,
                state,
                round_index=rnd + 1,
                total_rounds=cfg.global_rounds,
                training_aggregation='sample_weighted',
                server_optimizer='mean',
                federated_normalization='shared',
                local_norm_keys=[],
                quantizer=None,
                server_state=server_state,
            )
        else:
            state, meta, server_state = aggregate_federated(
                results,
                state,
                round_index=rnd + 1,
                total_rounds=cfg.global_rounds,
                training_aggregation=cfg.aggregation_mode,
                server_optimizer=cfg.server_optimizer,
                federated_normalization=cfg.federated_normalization,
                local_norm_keys=local_norm_keys,
                quantizer=quantizer,
                server_state=server_state,
                server_learning_rate=cfg.server_learning_rate,
                server_beta1=cfg.server_beta1,
                server_beta2=cfg.server_beta2,
                server_epsilon=cfg.server_epsilon,
            )

        total_bytes += meta['bytes_communicated']
        ev = ray.get(workers[0].evaluate.remote(state, 256))
        round_time_sec = time.time() - t0
        hist.append({
            'round': rnd + 1,
            'eval_accuracy': float(ev['accuracy']),
            'eval_loss': float(ev['loss']),
            'avg_worker_accuracy': float(meta['avg_worker_accuracy']),
            'avg_worker_loss': float(meta['avg_worker_loss']),
            'compression_ratio': float(meta['compression_ratio']),
            'bytes_communicated': int(meta['bytes_communicated']),
            'round_time_sec': round_time_sec,
        })
        print(
            f'  [{tag}] rnd={rnd+1:02d}/{cfg.global_rounds} '
            f'acc={ev["accuracy"]:.4f} '
            f'loss={ev["loss"]:.4f} '
            f'cr={meta["compression_ratio"]:.2f}x '
            f't={round_time_sec:.1f}s'
        )

    for worker_actor in workers:
        ray.kill(worker_actor)

    training_time_sec = float(sum(item['round_time_sec'] for item in hist)) if hist else 0.0
    avg_iter = training_time_sec / max(len(hist), 1)
    return {
        'experiment': tag,
        'dataset': ds_name,
        'final_accuracy': hist[-1]['eval_accuracy'] if hist else 0.0,
        'final_loss': hist[-1]['eval_loss'] if hist else 0.0,
        'training_time_sec': training_time_sec,
        'setup_time_sec': setup_time_sec,
        'ray_init_time_sec': ray_init_time_sec,
        'avg_iteration_time_sec': avg_iter,
        'median_iteration_time_sec': float(np.median([item['round_time_sec'] for item in hist])) if hist else 0.0,
        'total_runtime_sec': training_time_sec + setup_time_sec + ray_init_time_sec,
        'total_bytes_communicated': total_bytes,
        'peak_memory_mb': measure_peak_memory_mb(),
        'round_history': hist,
    }


## Track 1 — Distributed Baselines

In [ ]:
def run_centralized(ds_name, data_root, subset_size, nc,
                   num_epochs=20, lr=0.1, wd=5e-4, ls=0.1,
                   use_amp=True, seed=42, tag='centralized'):
    """Single model trained on the full dataset. Accuracy ceiling."""
    seed_everything(seed)
    _, _, tl, vl = make_loaders(ds_name, data_root, subset_size, seed=seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model  = SmallImageResNet18(num_classes=nc).to(device)
    if device.type == 'cuda':
        model = model.to(memory_format=torch.channels_last)
    opt, sch = make_sgd_cosine(model, lr, wd, num_epochs)
    crit     = nn.CrossEntropyLoss(label_smoothing=ls)
    scaler   = torch.cuda.amp.GradScaler() if (use_amp and device.type == 'cuda') else None
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    hist = []; started = time.time()
    for ep in range(num_epochs):
        t0 = time.time()
        tl_, vl_ = tl, vl   # re-iterate each epoch
        tr_loss, tr_acc = train_one_epoch(model, tl_, opt, crit, scaler, device)
        vl_loss, vl_acc = eval_model(model, vl_, crit, device)
        sch.step()
        hist.append({'epoch': ep+1, 'eval_accuracy': vl_acc, 'eval_loss': vl_loss,
                     'train_accuracy': tr_acc, 'epoch_time_sec': time.time()-t0})
        print(f'  [{tag}] ep={ep+1:02d}/{num_epochs} '
              f'train_acc={tr_acc:.4f} val_acc={vl_acc:.4f}')
    return {
        'experiment': tag, 'dataset': ds_name,
        'final_accuracy': hist[-1]['eval_accuracy'],
        'final_loss':     hist[-1]['eval_loss'],
        'training_time_sec': time.time()-started,
        'total_bytes_communicated': 0,
        'peak_memory_mb': measure_peak_memory_mb(),
        'epoch_history':  hist,
    }


# ── PyTorch DDP (single-node, single-GPU) ──────────────────────────────────
def run_ddp(ds_name, data_root, subset_size, nc,
            num_epochs=20, lr=0.1, wd=5e-4, ls=0.1,
            use_amp=True, seed=42, tag='pytorch_ddp'):
    """
    DDP on 1 GPU (world_size=1).
    Measures DDP wrapper overhead; on multi-GPU the all-reduce benefit would appear.
    Note: On single GPU DDP accuracy == centralized; the value here is framework overhead.
    """
    os.environ.setdefault('MASTER_ADDR', 'localhost')
    os.environ.setdefault('MASTER_PORT', '12355')
    if not dist.is_initialized():
        dist.init_process_group(backend='gloo', rank=0, world_size=1)
    seed_everything(seed)
    _, _, tl, vl = make_loaders(ds_name, data_root, subset_size, seed=seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    base   = SmallImageResNet18(num_classes=nc).to(device)
    if device.type == 'cuda':
        base = base.to(memory_format=torch.channels_last)
    model  = nn.parallel.DistributedDataParallel(
        base,
        device_ids=[torch.cuda.current_device()] if device.type == 'cuda' else None,
    )
    opt, sch = make_sgd_cosine(model, lr, wd, num_epochs)
    crit     = nn.CrossEntropyLoss(label_smoothing=ls)
    scaler   = torch.cuda.amp.GradScaler() if (use_amp and device.type == 'cuda') else None
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    hist = []; started = time.time()
    for ep in range(num_epochs):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, tl, opt, crit, scaler, device)
        vl_loss, vl_acc = eval_model(model, vl, crit, device)
        sch.step()
        hist.append({'epoch': ep+1, 'eval_accuracy': vl_acc, 'eval_loss': vl_loss,
                     'epoch_time_sec': time.time()-t0})
        print(f'  [{tag}] ep={ep+1:02d}/{num_epochs} val_acc={vl_acc:.4f}')
    try:
        dist.destroy_process_group()
    except Exception:
        pass
    return {
        'experiment': tag, 'dataset': ds_name,
        'final_accuracy': hist[-1]['eval_accuracy'],
        'final_loss':     hist[-1]['eval_loss'],
        'training_time_sec': time.time()-started,
        'total_bytes_communicated': 0,
        'peak_memory_mb': measure_peak_memory_mb(),
        'epoch_history':  hist,
        'note': 'Single-GPU DDP (world_size=1): overhead only, no all-reduce benefit',
    }


# ── PyTorch FSDP (single-node, single-GPU) ─────────────────────────────────
def run_fsdp(ds_name, data_root, subset_size, nc,
             num_epochs=20, lr=0.1, wd=5e-4, ls=0.1,
             use_amp=True, seed=42, tag='pytorch_fsdp'):
    """FSDP on 1 GPU. Shows FSDP framework overhead."""
    try:
        from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
        from torch.distributed.fsdp import ShardingStrategy, MixedPrecision
    except ImportError:
        return {'experiment': tag, 'dataset': ds_name, 'available': False,
                'reason': 'torch.distributed.fsdp not available'}
    os.environ.setdefault('MASTER_ADDR', 'localhost')
    os.environ.setdefault('MASTER_PORT', '12356')
    if not dist.is_initialized():
        dist.init_process_group(backend='gloo', rank=0, world_size=1)
    seed_everything(seed)
    _, _, tl, vl = make_loaders(ds_name, data_root, subset_size, seed=seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    base   = SmallImageResNet18(num_classes=nc)
    mp_pol = None
    if use_amp and device.type == 'cuda':
        mp_pol = MixedPrecision(param_dtype=torch.float16,
                                reduce_dtype=torch.float16,
                                buffer_dtype=torch.float16)
    model = FSDP(base,
                 sharding_strategy=ShardingStrategy.FULL_SHARD,
                 mixed_precision=mp_pol,
                 device_id=torch.cuda.current_device() if device.type == 'cuda' else None)
    opt, sch = make_sgd_cosine(model, lr, wd, num_epochs)
    crit     = nn.CrossEntropyLoss(label_smoothing=ls)
    scaler   = None  # FSDP handles mixed precision internally
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    hist = []; started = time.time()
    for ep in range(num_epochs):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, tl, opt, crit, scaler, device)
        vl_loss, vl_acc = eval_model(model, vl, crit, device)
        sch.step()
        hist.append({'epoch': ep+1, 'eval_accuracy': vl_acc, 'eval_loss': vl_loss,
                     'epoch_time_sec': time.time()-t0})
        print(f'  [{tag}] ep={ep+1:02d}/{num_epochs} val_acc={vl_acc:.4f}')
    try:
        dist.destroy_process_group()
    except Exception:
        pass
    return {
        'experiment': tag, 'dataset': ds_name,
        'final_accuracy': hist[-1]['eval_accuracy'],
        'final_loss':     hist[-1]['eval_loss'],
        'training_time_sec': time.time()-started,
        'total_bytes_communicated': 0,
        'peak_memory_mb': measure_peak_memory_mb(),
        'epoch_history':  hist,
        'note': 'Single-GPU FSDP (world_size=1): overhead only',
    }


## Track 2 — Ensemble Baselines (TorchEnsemble)

In [ ]:
def run_torchensemble(ds_name, data_root, subset_size, nc,
                     n_estimators=4, etype='voting',
                     num_epochs=20, lr=0.1, wd=5e-4,
                     seed=42, tag=None):
    """TorchEnsemble VotingClassifier or BaggingClassifier with n_estimators=4."""
    tag = tag or f'torchensemble_{etype}'
    try:
        from torchensemble import VotingClassifier, BaggingClassifier
    except ImportError:
        return {'experiment': tag, 'dataset': ds_name, 'available': False,
                'reason': 'pip install torchensemble'}
    seed_everything(seed)
    _, _, tl, vl = make_loaders(ds_name, data_root, subset_size, seed=seed)
    Cls  = VotingClassifier if etype == 'voting' else BaggingClassifier
    ens  = Cls(
        estimator=SmallImageResNet18,
        n_estimators=n_estimators,
        estimator_args={'num_classes': nc},
        cuda=torch.cuda.is_available(),
    )
    ens.set_optimizer('SGD', lr=lr, momentum=0.9, weight_decay=wd, nesterov=True)
    ens.set_scheduler('CosineAnnealingLR', T_max=num_epochs, eta_min=1e-6)
    ens.set_criterion(nn.CrossEntropyLoss(label_smoothing=0.1))
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    started = time.time()
    ens.fit(tl, epochs=num_epochs, test_loader=vl, save_model=False)
    acc = float(ens.evaluate(vl))
    return {
        'experiment': tag, 'dataset': ds_name,
        'final_accuracy': acc,
        'final_loss':     None,
        'training_time_sec': time.time()-started,
        'total_bytes_communicated': 0,
        'peak_memory_mb': measure_peak_memory_mb(),
        'n_estimators':  n_estimators,
        'available':     True,
    }


def run_hqde_independent(ds_name, data_root, subset_size, nc,
                         n_workers=4, num_epochs=20, seed=42,
                         tag='hqde_independent_replicate'):
    """
    HQDE in independent + replicate mode.
    Workers train on the full dataset independently; predictions are averaged.
    Directly comparable to TorchEnsemble (same number of estimators).
    """
    try:
        from hqde import create_hqde_system
    except ImportError:
        return {'experiment': tag, 'dataset': ds_name, 'available': False,
                'reason': 'hqde not installed'}
    seed_everything(seed)
    _, _, tl, vl = make_loaders(ds_name, data_root, subset_size, seed=seed)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    started = time.time()
    sys = create_hqde_system(
        SmallImageResNet18, {'num_classes': nc},
        num_workers=n_workers,
        training_config={
            'ensemble_mode':           'independent',
            'batch_assignment':        'replicate',
            'optimizer':               'sgd',
            'learning_rate':           0.1,
            'weight_decay':            5e-4,
            'label_smoothing':         0.1,
            'use_amp':                 True,
            'prediction_aggregation':  'mean',
        },
    )
    metrics = sys.train(tl, num_epochs=num_epochs)
    # Collect predictions from all batches
    all_preds, all_ys = [], []
    for xb, yb in vl:
        preds = sys.predict([xb])
        all_preds.append(preds)
        all_ys.append(yb)
    preds_cat = torch.cat(all_preds, 0)
    ys_cat    = torch.cat(all_ys, 0)
    acc = float((preds_cat.argmax(1) == ys_cat).float().mean().item())
    sys.cleanup()
    return {
        'experiment': tag, 'dataset': ds_name,
        'final_accuracy': acc,
        'final_loss':     metrics.get('final_loss'),
        'training_time_sec': time.time()-started,
        'total_bytes_communicated': 0,
        'peak_memory_mb': measure_peak_memory_mb(),
        'available': True,
    }


## Track 3 — Federated Learning Baselines (Flower)

In [ ]:
def run_flower(ds_name, data_root, subset_size, nc,
               num_clients=12, fraction_fit=0.33,
               num_rounds=20, local_epochs=1,
               lr=0.1, wd=5e-4, ls=0.1, use_amp=True,
               partition_mode='dirichlet', alpha=0.5, seed=42,
               fedprox_mu=0.0, tag='flower_fedavg'):
    """
    Flower FedAvg (fedprox_mu=0) or FedProx (fedprox_mu>0) via simulation.
    Matches HQDE: same backbone, same num_clients, same local_epochs, same partition.
    """
    try:
        import flwr as fl
    except ImportError:
        return {'experiment': tag, 'dataset': ds_name, 'available': False,
                'reason': 'pip install flwr'}

    # Shut down any existing Ray from HQDE before Flower starts its own
    if ray.is_initialized():
        ray.shutdown()

    seed_everything(seed)
    device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    train_ds  = load_raw_dataset(ds_name, data_root, True, subset_size, seed)
    test_ds   = load_raw_dataset(ds_name, data_root, False, None, seed)
    parts     = partition_indices(get_labels(train_ds), num_clients,
                                  partition_mode, alpha, seed)
    test_loader = DataLoader(test_ds, batch_size=256, shuffle=False,
                             num_workers=2, pin_memory=True)
    init_model  = SmallImageResNet18(num_classes=nc)
    init_params = [v.detach().numpy() for v in init_model.state_dict().values()]
    param_keys  = list(init_model.state_dict().keys())
    del init_model
    round_hist  = []

    class _Client(fl.client.NumPyClient):
        def __init__(self, cid):
            self.cid    = int(cid)
            self.shard  = parts[self.cid]
            self.model  = SmallImageResNet18(num_classes=nc).to(device)
            self.loader = DataLoader(
                Subset(train_ds, self.shard),
                batch_size=128, shuffle=True, num_workers=2, pin_memory=True,
            )

        def _set(self, params):
            self.model.load_state_dict(
                {k: torch.tensor(v) for k, v in zip(param_keys, params)}, strict=True
            )

        def get_parameters(self, config):
            return [v.detach().cpu().numpy() for v in self.model.state_dict().values()]

        def fit(self, parameters, config):
            self._set(parameters)
            # Store global params for FedProx proximal term
            global_params = [p.detach().clone() for p in self.model.parameters()]
            if device.type == 'cuda':
                self.model = self.model.to(memory_format=torch.channels_last)
            opt    = torch.optim.SGD(self.model.parameters(), lr=lr,
                                     momentum=0.9, weight_decay=wd, nesterov=True)
            crit   = nn.CrossEntropyLoss(label_smoothing=float(ls))
            scaler = (torch.cuda.amp.GradScaler()
                      if (use_amp and device.type == 'cuda') else None)
            tot_n = 0
            self.model.train()
            for _ in range(local_epochs):
                for xb, yb in self.loader:
                    xb = _move(xb, device)
                    yb = yb.to(device, non_blocking=True)
                    opt.zero_grad(set_to_none=True)
                    if scaler:
                        with torch.autocast('cuda', dtype=torch.float16):
                            out  = self.model(xb); loss = crit(out, yb)
                        if fedprox_mu > 0:
                            prox = sum(
                                (p - gp.to(device)).norm() ** 2
                                for p, gp in zip(self.model.parameters(), global_params)
                            )
                            loss = loss + (fedprox_mu / 2) * prox
                        scaler.scale(loss).backward()
                        scaler.unscale_(opt)
                        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                        scaler.step(opt); scaler.update()
                    else:
                        out  = self.model(xb); loss = crit(out, yb)
                        if fedprox_mu > 0:
                            prox = sum(
                                (p - gp.to(device)).norm() ** 2
                                for p, gp in zip(self.model.parameters(), global_params)
                            )
                            loss = loss + (fedprox_mu / 2) * prox
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                        opt.step()
                    tot_n += int(yb.size(0))
            return self.get_parameters({}), tot_n, {}

        def evaluate(self, parameters, config):
            self._set(parameters)
            loss, acc = eval_model(self.model, self.loader, nn.CrossEntropyLoss(), device)
            return float(loss), len(self.shard), {'accuracy': float(acc)}

    def evaluate_fn(server_round, parameters, config):
        model = SmallImageResNet18(num_classes=nc).to(device)
        model.load_state_dict({k: torch.tensor(v) for k, v in zip(param_keys, parameters)})
        loss, acc = eval_model(model, test_loader, nn.CrossEntropyLoss(), device)
        round_hist.append({'round': server_round, 'eval_accuracy': acc, 'eval_loss': loss})
        print(f'  [{tag}] rnd={server_round:02d}/{num_rounds} acc={acc:.4f} loss={loss:.4f}')
        return float(loss), {'accuracy': float(acc)}

    strategy = fl.server.strategy.FedAvg(
        fraction_fit=float(fraction_fit),
        fraction_evaluate=0.0,
        min_fit_clients=max(1, int(num_clients * fraction_fit)),
        min_available_clients=num_clients,
        evaluate_fn=evaluate_fn,
        initial_parameters=fl.common.ndarrays_to_parameters(init_params),
    )
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    started = time.time()
    fl.simulation.start_simulation(
        client_fn=lambda cid: _Client(cid).to_client(),
        num_clients=num_clients,
        config=fl.server.ServerConfig(num_rounds=num_rounds),
        strategy=strategy,
        client_resources={
            'num_gpus':  0.25 if torch.cuda.is_available() else 0.0,
            'num_cpus':  1,
        },
        ray_init_args={'ignore_reinit_error': True, 'include_dashboard': False},
    )
    param_size_bytes = sum(
        torch.tensor(p).numel() * 4 for p in init_params
        if torch.is_floating_point(torch.tensor(p))
    )
    selected   = max(1, int(num_clients * fraction_fit))
    comm_bytes = param_size_bytes * selected * 2 * num_rounds
    return {
        'experiment': tag, 'dataset': ds_name,
        'final_accuracy': round_hist[-1]['eval_accuracy'] if round_hist else 0.0,
        'final_loss':     round_hist[-1]['eval_loss']     if round_hist else 0.0,
        'training_time_sec': time.time()-started,
        'total_bytes_communicated': comm_bytes,
        'peak_memory_mb': measure_peak_memory_mb(),
        'round_history':  round_hist,
        'available':      True,
    }


def run_flower_fedprox(ds_name, data_root, subset_size, nc, mu=0.1, **kwargs):
    kwargs.setdefault('tag', f'flower_fedprox_mu{mu}')
    return run_flower(ds_name, data_root, subset_size, nc, fedprox_mu=mu, **kwargs)


## Track 4 — HQDE Ablations

| Ablation | Config delta |
|---|---|
| `hqde_effweighted_q`   | efficiency_weighted + quantization ON  (full HQDE) |
| `hqde_effweighted_noq` | efficiency_weighted + quantization OFF |
| `hqde_mean_noq`        | plain sample-weighted mean + quantization OFF (= vanilla FedAvg) |
| `hqde_iid`             | IID partition instead of Dirichlet |
| `hqde_hard_noniid`     | Dirichlet alpha=0.1 (harder non-IID) |


In [ ]:
def run_hqde_ablation(ds_name, data_root, subset_size, nc,
                     base_cfg: ScheduledHQDEConfig,
                     overrides: dict, tag: str):
    cfg = copy.deepcopy(base_cfg)
    for k, v in overrides.items():
        setattr(cfg, k, v)
    return run_hqde_scheduled(ds_name, data_root, subset_size, nc, cfg, tag)


ABLATIONS = [
    # (overrides_dict,  tag)
    ({'aggregation_mode': 'efficiency_weighted', 'use_quantization': True},
     'hqde_effweighted_q'),
    ({'aggregation_mode': 'efficiency_weighted', 'use_quantization': False},
     'hqde_effweighted_noq'),
    ({'aggregation_mode': 'fedavg_uniform',      'use_quantization': False},
     'hqde_mean_noq'),
    ({'partition_mode': 'iid',                   'use_quantization': True},
     'hqde_iid'),
    ({'dirichlet_alpha': 0.1,                    'use_quantization': True},
     'hqde_hard_noniid'),
]


## Experiment Configuration & Runner

In [ ]:
SUBSET_SIZE = 20000
NUM_ROUNDS = 20
NUM_EPOCHS = 20
SEEDS = [42]

DATASETS = [
    ('cifar10', 10),
    ('cifar100', 100),
]

BASE_CFG = ScheduledHQDEConfig(
    logical_workers=12,
    active_gpu_executors=4,
    gpu_fraction=0.25,
    local_epochs=1,
    global_rounds=NUM_ROUNDS,
    learning_rate=0.1,
    weight_decay=5e-4,
    label_smoothing=0.1,
    use_amp=True,
    aggregation_mode='sample_weighted',
    server_optimizer='fedadam',
    federated_normalization='local_bn',
    server_learning_rate=1.0,
    server_beta1=0.9,
    server_beta2=0.99,
    server_epsilon=1e-3,
    use_quantization=False,
    quantization_config={
        'base_bits': 12,
        'min_bits': 8,
        'max_bits': 16,
        'warmup_rounds': 8,
        'block_size': 1024,
    },
    partition_mode='dirichlet',
    dirichlet_alpha=0.5,
    seed=42,
)
print('Config ready. Datasets:', [d[0] for d in DATASETS])
print(f'Budget: {NUM_ROUNDS} rounds / {NUM_EPOCHS} epochs, subset={SUBSET_SIZE}')
print('HQDE scheduled default: sample-weighted + FedAdam + local BN, quantization off')


## Run All Experiments

In [ ]:
all_results: Dict[str, Dict] = {}

for ds_name, nc in DATASETS:
    all_results[ds_name] = {}
    print(f'\n{"="*68}')
    print(f' Dataset: {ds_name.upper()}  ({nc} classes)')
    print(f'{"="*68}')

    # ── Track 1: Distributed ───────────────────────────────────────────────
    print('\n── Track 1: Distributed ──────────────────────────────────────')

    print('\n[1/5] Centralized (upper bound)')
    r = run_centralized(ds_name, DATA_ROOT, SUBSET_SIZE, nc,
                        num_epochs=NUM_EPOCHS, seed=42)
    all_results[ds_name]['centralized'] = r
    save_json(r, OUT / f'{ds_name}_centralized.json')
    print(f'  → final_acc={r["final_accuracy"]:.4f}')

    print('\n[2/5] HQDE scheduled (sample_weighted + FedAdam + local_bn)')
    ensure_ray()
    r = run_hqde_scheduled(ds_name, DATA_ROOT, SUBSET_SIZE, nc,
                           BASE_CFG, tag='hqde_scheduled')
    all_results[ds_name]['hqde_scheduled'] = r
    save_json(r, OUT / f'{ds_name}_hqde_scheduled.json')
    print(f'  → final_acc={r["final_accuracy"]:.4f}')

    print('\n[3/5] Vanilla FedAvg baseline (uniform aggregation, no quantization)')
    fedavg_cfg = copy.deepcopy(BASE_CFG)
    fedavg_cfg.aggregation_mode  = 'fedavg_uniform'
    fedavg_cfg.use_quantization  = False
    r = run_hqde_scheduled(ds_name, DATA_ROOT, SUBSET_SIZE, nc,
                           fedavg_cfg, tag='vanilla_fedavg')
    all_results[ds_name]['vanilla_fedavg'] = r
    save_json(r, OUT / f'{ds_name}_vanilla_fedavg.json')
    print(f'  → final_acc={r["final_accuracy"]:.4f}')

    print('\n[4/5] PyTorch DDP (single GPU)')
    try:
        r = run_ddp(ds_name, DATA_ROOT, SUBSET_SIZE, nc,
                    num_epochs=NUM_EPOCHS, seed=42)
        all_results[ds_name]['pytorch_ddp'] = r
        save_json(r, OUT / f'{ds_name}_pytorch_ddp.json')
        print(f'  → final_acc={r["final_accuracy"]:.4f}')
    except Exception as exc:
        print(f'  ! DDP failed: {exc}')
        all_results[ds_name]['pytorch_ddp'] = {'available': False, 'reason': str(exc)}

    print('\n[5/5] PyTorch FSDP (single GPU)')
    try:
        r = run_fsdp(ds_name, DATA_ROOT, SUBSET_SIZE, nc,
                     num_epochs=NUM_EPOCHS, seed=42)
        all_results[ds_name]['pytorch_fsdp'] = r
        save_json(r, OUT / f'{ds_name}_pytorch_fsdp.json')
        print(f'  → final_acc={r["final_accuracy"]:.4f}')
    except Exception as exc:
        print(f'  ! FSDP failed: {exc}')
        all_results[ds_name]['pytorch_fsdp'] = {'available': False, 'reason': str(exc)}

    # ── Track 2: Ensemble ──────────────────────────────────────────────────
    print('\n── Track 2: Ensemble ─────────────────────────────────────────')

    for etype in ['voting', 'bagging']:
        tag = f'torchensemble_{etype}'
        print(f'\nTorchEnsemble {etype} (n=4)')
        try:
            r = run_torchensemble(ds_name, DATA_ROOT, SUBSET_SIZE, nc,
                                  n_estimators=4, etype=etype,
                                  num_epochs=NUM_EPOCHS, seed=42, tag=tag)
            all_results[ds_name][tag] = r
            save_json(r, OUT / f'{ds_name}_{tag}.json')
            if r.get('available', True):
                print(f'  → final_acc={r["final_accuracy"]:.4f}')
        except Exception as exc:
            print(f'  ! {tag} failed: {exc}')
            all_results[ds_name][tag] = {'available': False, 'reason': str(exc)}

    print('\nHQDE independent+replicate (n_workers=4)')
    try:
        r = run_hqde_independent(ds_name, DATA_ROOT, SUBSET_SIZE, nc,
                                 n_workers=4, num_epochs=NUM_EPOCHS, seed=42)
        all_results[ds_name]['hqde_independent'] = r
        save_json(r, OUT / f'{ds_name}_hqde_independent.json')
        if r.get('available', True):
            print(f'  → final_acc={r["final_accuracy"]:.4f}')
    except Exception as exc:
        print(f'  ! HQDE independent failed: {exc}')
        all_results[ds_name]['hqde_independent'] = {'available': False, 'reason': str(exc)}

    # ── Track 3: Federated (Flower) ────────────────────────────────────────
    print('\n── Track 3: Federated ────────────────────────────────────────')
    fl_kwargs = dict(
        num_clients=12, fraction_fit=0.33,
        num_rounds=NUM_ROUNDS, local_epochs=1,
        lr=0.1, wd=5e-4, ls=0.1, use_amp=True,
        partition_mode='dirichlet', alpha=0.5, seed=42,
    )

    print('\nFlower FedAvg')
    try:
        r = run_flower(ds_name, DATA_ROOT, SUBSET_SIZE, nc,
                       **fl_kwargs, fedprox_mu=0.0, tag='flower_fedavg')
        all_results[ds_name]['flower_fedavg'] = r
        save_json(r, OUT / f'{ds_name}_flower_fedavg.json')
        if r.get('available', True):
            print(f'  → final_acc={r["final_accuracy"]:.4f}')
    except Exception as exc:
        print(f'  ! Flower FedAvg failed: {exc}')
        all_results[ds_name]['flower_fedavg'] = {'available': False, 'reason': str(exc)}

    print('\nFlower FedProx (mu=0.1)')
    try:
        r = run_flower(ds_name, DATA_ROOT, SUBSET_SIZE, nc,
                       **fl_kwargs, fedprox_mu=0.1, tag='flower_fedprox_mu0.1')
        all_results[ds_name]['flower_fedprox'] = r
        save_json(r, OUT / f'{ds_name}_flower_fedprox.json')
        if r.get('available', True):
            print(f'  → final_acc={r["final_accuracy"]:.4f}')
    except Exception as exc:
        print(f'  ! Flower FedProx failed: {exc}')
        all_results[ds_name]['flower_fedprox'] = {'available': False, 'reason': str(exc)}

    # Restart Ray for subsequent HQDE experiments
    if not ray.is_initialized():
        ensure_ray()

    # ── Track 4: HQDE Ablations ────────────────────────────────────────────
    print('\n── Track 4: HQDE Ablations ───────────────────────────────────')
    for overrides, abl_tag in ABLATIONS:
        print(f'\nAblation: {abl_tag}')
        try:
            r = run_hqde_ablation(ds_name, DATA_ROOT, SUBSET_SIZE, nc,
                                  BASE_CFG, overrides, abl_tag)
            all_results[ds_name][f'abl_{abl_tag}'] = r
            save_json(r, OUT / f'{ds_name}_abl_{abl_tag}.json')
            print(f'  → final_acc={r["final_accuracy"]:.4f}')
        except Exception as exc:
            print(f'  ! {abl_tag} failed: {exc}')
            all_results[ds_name][f'abl_{abl_tag}'] = {'available': False, 'reason': str(exc)}

save_json(all_results, OUT / 'all_results.json')
print('\n✓ All experiments complete. Raw results saved.')


## Results Tables

In [ ]:
def build_table(all_results) -> pd.DataFrame:
    rows = []
    for ds, exps in all_results.items():
        for key, res in exps.items():
            if not res.get('available', True) or 'final_accuracy' not in res:
                continue
            acc = res.get('final_accuracy') or 0.0
            rows.append({
                'dataset': ds,
                'track': ('ablation' if key.startswith('abl_') else
                          'distributed' if key in ('centralized', 'pytorch_ddp', 'pytorch_fsdp', 'hqde_scheduled', 'vanilla_fedavg') else
                          'ensemble' if 'ensemble' in key or 'independent' in key else
                          'federated'),
                'experiment': res.get('experiment', key),
                'accuracy': round(float(acc), 4),
                'time_sec': round(float(res.get('training_time_sec') or 0), 1),
                'setup_sec': round(float(res.get('setup_time_sec') or 0), 1),
                'avg_iter_sec': round(float(res.get('avg_iteration_time_sec') or 0), 2),
                'ray_init_sec': round(float(res.get('ray_init_time_sec') or 0), 1),
                'total_sec': round(float(res.get('total_runtime_sec') or 0), 1),
                'mem_MB': round(float(res.get('peak_memory_mb') or 0), 1),
                'comm_MB': round(float(res.get('total_bytes_communicated') or 0) / 1024**2, 2),
            })
    return pd.DataFrame(rows).sort_values(['dataset', 'track', 'accuracy'],
                                          ascending=[True, True, False])

summary = build_table(all_results)
pd.set_option('display.max_rows', 80)
pd.set_option('display.max_colwidth', 40)
print('\n' + '=' * 80)
print('COMPREHENSIVE BENCHMARK - SUMMARY TABLE')
print('=' * 80)
display(summary)
save_json(summary.to_dict('records'), OUT / 'summary_table.json')


In [ ]:
# Per-track pivot for paper tables
for track in ['distributed', 'ensemble', 'federated', 'ablation']:
    sub = summary[summary['track'] == track]
    if sub.empty:
        continue
    print(f'\n── {track.upper()} ──────────────────────────────────────────────')
    pivot = sub.pivot_table(
        index='experiment',
        columns='dataset',
        values='accuracy',
        aggfunc='first',
    ).round(4)
    display(pivot)
    pivot.to_csv(OUT / f'table_{track}.csv')


## Figures

In [ ]:
fig_kwargs = dict(dpi=150, bbox_inches='tight')

for ds in summary['dataset'].unique():
    df = summary[summary['dataset'] == ds].copy()
    df = df[~df['experiment'].str.startswith('hqde_abl') &
            ~df['experiment'].str.startswith('abl')]
    df = df.sort_values('accuracy', ascending=True)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Benchmark - {ds.upper()}', fontsize=14, fontweight='bold')

    PALETTE = {
        'centralized': '#1a237e',
        'pytorch_ddp': '#283593',
        'pytorch_fsdp': '#3949ab',
        'hqde_scheduled': '#e53935',
        'vanilla_fedavg': '#ff7043',
        'flower_fedavg': '#2e7d32',
        'flower_fedprox': '#43a047',
        'torchensemble_voting': '#6a1b9a',
        'torchensemble_bagging': '#8e24aa',
        'hqde_independent_replicate': '#c62828',
    }
    colors = [PALETTE.get(e, '#78909c') for e in df['experiment']]

    ax = axes[0]
    ax.barh(df['experiment'], df['accuracy'], color=colors, alpha=0.85)
    ax.set_xlabel('Final Accuracy')
    ax.set_title('Accuracy')
    ax.set_xlim([0, 1])
    for i, (_, row) in enumerate(df.iterrows()):
        ax.text(row['accuracy'] + 0.005, i, f'{row["accuracy"]:.4f}', va='center', fontsize=8)

    ax = axes[1]
    time_df = df.sort_values('time_sec')
    ax.barh(time_df['experiment'], time_df['time_sec'], color='steelblue', alpha=0.8)
    ax.set_xlabel('Pure Training Time (s, excl. Ray init)')
    ax.set_title('Training Time')

    comm = df[df['comm_MB'] > 0].sort_values('comm_MB')
    ax = axes[2]
    if not comm.empty:
        ax.barh(comm['experiment'], comm['comm_MB'], color='coral', alpha=0.8)
        ax.set_xlabel('Communication (MB)')
        ax.set_title('Communication Cost')
    else:
        ax.text(0.5, 0.5, 'No communication\n(local training)', ha='center',
                va='center', transform=ax.transAxes, fontsize=11)
        ax.set_title('Communication Cost')

    plt.tight_layout()
    plt.savefig(OUT / f'fig1_{ds}_comparison.png', **fig_kwargs)
    plt.show()


In [ ]:
# ── Figure 2: convergence curves (federated experiments) ──────────────────
FED_EXPS = {
    'hqde_scheduled':  ('#e53935', 'HQDE efficiency-weighted'),
    'vanilla_fedavg':  ('#ff7043', 'Vanilla FedAvg'),
    'flower_fedavg':   ('#2e7d32', 'Flower FedAvg'),
    'flower_fedprox':  ('#43a047', 'Flower FedProx'),
}

for ds, exps in all_results.items():
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Convergence — {ds.upper()}  (Dirichlet α=0.5)', fontsize=13)

    for key, (color, label) in FED_EXPS.items():
        res  = exps.get(key, {})
        hist = res.get('round_history', [])
        if not hist:
            continue
        rds  = [h['round']         for h in hist]
        accs = [h['eval_accuracy'] for h in hist]
        loss = [h['eval_loss']     for h in hist]
        a1.plot(rds, accs, marker='o', ms=3, lw=2, color=color, label=label)
        a2.plot(rds, loss, marker='o', ms=3, lw=2, color=color, label=label)

    cent = exps.get('centralized', {})
    if cent.get('final_accuracy'):
        a1.axhline(cent['final_accuracy'], ls='--', lw=1.5, color='black',
                   label=f'Centralized ({cent["final_accuracy"]:.4f})')

    a1.set_xlabel('Round'); a1.set_ylabel('Accuracy'); a1.set_title('Eval Accuracy vs Round')
    a2.set_xlabel('Round'); a2.set_ylabel('Loss');     a2.set_title('Eval Loss vs Round')
    for ax in (a1, a2):
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT / f'fig2_{ds}_convergence.png', **fig_kwargs)
    plt.show()


In [ ]:
# ── Figure 3: HQDE ablation heatmaps ────────────────────────────────────
for ds, exps in all_results.items():
    abl_rows = []
    for key, res in exps.items():
        if not key.startswith('abl_') or not res.get('available', True):
            continue
        abl_rows.append({
            'variant':  key.replace('abl_', ''),
            'accuracy': round(float(res.get('final_accuracy') or 0), 4),
            'time_sec': round(float(res.get('training_time_sec') or 0), 0),
            'comm_MB':  round(float(res.get('total_bytes_communicated') or 0)/1024**2, 1),
        })
    if not abl_rows:
        continue
    abl_df = pd.DataFrame(abl_rows).set_index('variant')
    fig, axes = plt.subplots(1, 2, figsize=(12, max(3, len(abl_rows))))
    fig.suptitle(f'HQDE Ablations — {ds.upper()}', fontsize=13)
    sns.heatmap(abl_df[['accuracy']], annot=True, fmt='.4f', cmap='YlGnBu',
                linewidths=0.5, ax=axes[0], cbar=True)
    axes[0].set_title('Final Accuracy'); axes[0].set_ylabel('')
    sns.heatmap(abl_df[['comm_MB']], annot=True, fmt='.1f', cmap='OrRd',
                linewidths=0.5, ax=axes[1], cbar=True)
    axes[1].set_title('Communication (MB)'); axes[1].set_ylabel('')
    plt.tight_layout()
    plt.savefig(OUT / f'fig3_{ds}_ablation.png', **fig_kwargs)
    plt.show()


In [ ]:
# ── Figure 4: accuracy vs communication scatter ─────────────────────────
for ds, exps in all_results.items():
    pts = []
    for key, res in exps.items():
        if key.startswith('abl_') or not res.get('available', True):
            continue
        comm = float(res.get('total_bytes_communicated') or 0) / 1024**2
        acc  = float(res.get('final_accuracy') or 0)
        if acc > 0:
            pts.append({'label': res.get('experiment', key), 'comm_MB': comm, 'accuracy': acc})
    if not pts:
        continue
    fig, ax = plt.subplots(figsize=(9, 5))
    for p in pts:
        ax.scatter(max(p['comm_MB'], 0.01), p['accuracy'], s=120, zorder=5)
        ax.annotate(p['label'], (max(p['comm_MB'], 0.01), p['accuracy']),
                    xytext=(5, 3), textcoords='offset points', fontsize=8)
    ax.set_xscale('symlog', linthresh=1)
    ax.set_xlabel('Communication Cost (MB, symlog scale)')
    ax.set_ylabel('Final Accuracy')
    ax.set_title(f'Accuracy vs Communication — {ds.upper()}')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT / f'fig4_{ds}_acc_vs_comm.png', **fig_kwargs)
    plt.show()


In [ ]:
save_json({'summary': summary.to_dict('records'), 'all_results': all_results},
          OUT / 'combined_summary.json')
if ray.is_initialized():
    ray.shutdown()
print(f'All outputs saved to {OUT}')
print('Files:')
for f in sorted(OUT.glob('*')):
    print(f'  {f.name}')
